# Run a baseline active-learning sampler

One of the 11 published baselines, one dataset, the full budget sweep. No
CellViT, no VLM, no text -- every baseline here selects on the frozen DINOv2
visual cache alone (`sampling.specs.BASELINE_SAMPLERS`). `scalpel`, this
project's own method, runs in `run_al_main.ipynb` instead.

Attach the Kaggle Dataset that `extract_visual_features.ipynb` published for
this (dataset, seed); this notebook does not extract features itself unless
that cache is missing, and extracting 90k images on the fly is a cost this
notebook is not meant to pay.

On a Kaggle **T4 x2** session the work is split across both GPUs, one worker
process per card. With several (seed, variant) jobs they are split job-wise;
with a single variant the BUDGET LIST is split instead (`SPLIT_BUDGETS`), which
is what keeps both cards busy on the common case of one sampler, one seed. A
prefix-exact sampler (`random`, `coreset`, `tcm`) derives its whole sweep from
one selection pass and so stays on one GPU by design. Set `PARALLEL = False`
to force serial.

Per budget this writes selected indices with their per-step acquisition trace
-- score, and wherever the method computes them separately, its uncertainty
and coverage terms -- plus the probe weights, the test predictions, the
metrics table, a sanity report and a run log. See `main.py`.

Metrics here are accuracy, precision, recall and macro-F1 only. PALM and any
other curve-level metric belong to `evaluate_al_sampler.ipynb`, which reads
these files.

The last cell packs everything into ONE zip at the top of `/kaggle/working`,
named `{dataset}_{sampler}_seed{seeds}`, and deletes the loose checkpoints --
the same shape both extraction notebooks use, because the Output tab is the
only way a file leaves a "Save & Run All" session.

Re-running the notebook skips variants whose `_results.pt` already exists, so a
session that hits the 12-hour limit can simply be run again.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
#
# ONE configuration per run. Every variable below takes a single value, never a
# list: the notebook ends in exactly one zip whose name states the whole
# configuration, and a run covering several could not name itself. To sweep
# seeds or samplers, run the notebook again with this cell changed.
#
# This costs nothing in speed -- both GPUs are used by splitting the BUDGET
# list (below), not by bundling several configurations into one session.
DATASET = "pathmnist"
SEED = 42

# ONE of the 11 published baselines. `scalpel` is excluded on purpose --
# see sampling.specs.BASELINE_SAMPLERS and run_al_main.ipynb.
#   random | coreset | typiclust | activeft | tcm
#   margin | entropy | badge | dropquery | uncertainty_herding | refine
SAMPLER = "uncertainty_herding"

# Override config.yaml for this sampler, e.g. {"k_nn": 10} for typiclust.
# Leave {} to run config.yaml as-is, which is what every baseline wants by
# default. Tuning a baseline's axes is baseline TUNING: either do it for every
# method or none, and say which in the report.
OVERRIDES = {}

# Leave None to derive the run name from the sampler and its config.
RUN_NAME = None

# Use both T4s when the session has them.
PARALLEL = True

# Split this run's budget list across both GPUs. This is what makes a single
# configuration use both cards.
#
# It applies ONLY to a sampler that is not prefix-exact, where every budget is
# an independent run anyway (margin, entropy, badge, dropquery,
# uncertainty_herding, refine, typiclust, activeft). A prefix-exact sampler
# (random, coreset, tcm) derives its whole sweep from ONE selection pass, so
# sharding would repeat that pass per shard and cost more than it saves -- the
# dispatch cell detects this and leaves it on one GPU. Nothing is lost by that:
# those three are also the cheap ones.
SPLIT_BUDGETS = True

# ---- TWO Kaggle Datasets, both required ----
#
# They are not interchangeable, and attaching only the feature one is the easy
# mistake to make: the feature cache holds the DINOv2 matrices and nothing
# else. The labels a sampler selects on, the labels the probe trains against,
# and the sample-order fingerprint that VALIDATES the cache itself all come
# from the raw images -- so `main.run` opens the dataset either way, and a
# missing DATA_ROOT fails before the cache is ever reached.
#
#   DATA_ROOT   raw images (pathmnist_224.npz, HistoSet, SkinTissue)
#   FEATURE_DIR the .npy features extract_visual_features.ipynb published,
#               which is what lets this notebook skip the backbone pass
#
# Both are starting points: the next cells search from here and print what
# they actually resolved to, so a remounted slug still works.
DATA_ROOT = "/kaggle/input/datasets/cryandrrich/nckh2026"
FEATURE_DIR = "/kaggle/input/datasets/nhtquyn/pathoactive"
OUTPUT_DIR = "/kaggle/working/checkpoints"

# Where a .npz dataset is re-exported as memory-mappable .npy files.
#
# An AL run reads no pixels -- it opens the dataset for labels, sample IDs and
# the cache fingerprint -- but NPZDataset reads a .npz EAGERLY, and
# PathMNIST-224 is ~15 GiB of uint8. Two GPU workers each holding their own
# copy is ~30 GiB, which is the whole Kaggle session, so one is killed before
# the sampler starts. Mapping costs scratch disk once and nothing per worker.
# Ignored for ImageFolder datasets (histoset, skintissue), which read per file.
MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading facebook/dinov2-base ...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import yaml
import torch

import main
from sampling.specs import BASELINE_SAMPLERS, spec_for
from utils.parallel import run_variants_parallel, visible_gpu_count
from utils.kaggle import find_data_root, find_visual_cache
from utils.progress import format_duration

In [ ]:
# `find_data_root` searches the default Kaggle mount points too, so a
# DATA_ROOT that was remounted under a different slug is still found. It
# reports where it actually landed rather than assuming.
DATA_ROOT = find_data_root([Path(DATA_ROOT)])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root (raw images):", DATA_ROOT)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

# This notebook is baseline-only. A sampler that needs a CellViT cell view or a
# VLM text prior belongs in run_al_main.ipynb instead -- catching that here, in
# an assert message that names the right notebook, beats a mid-run failure deep
# inside main.py once one budget has already spent GPU time.
assert SAMPLER in BASELINE_SAMPLERS, (
    f"{SAMPLER!r} is not a baseline (BASELINE_SAMPLERS={sorted(BASELINE_SAMPLERS)}). "
    "scalpel and any encoder/text variant run in run_al_main.ipynb."
)
spec = spec_for(SAMPLER)
assert "cell_embeddings" not in spec.needs, f"{SAMPLER} needs a CellViT cache -- unexpected for a baseline"
assert "text_embeddings" not in spec.needs, f"{SAMPLER} needs a VLM text prior -- unexpected for a baseline"

# One configuration per run: a list here would produce several results under
# one archive name, which could then not say which result was which.
assert isinstance(SEED, int), "SEED is one seed, not a list -- re-run the notebook to sweep"
assert isinstance(OVERRIDES, dict), "OVERRIDES is one config dict, not a list of variants"

dataset_info = config["datasets"][DATASET]
training_cfg = config["training"]
sampler_cfg = {**config.get("samplers", {}).get(SAMPLER, {}), **OVERRIDES}

data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"

# The DINOv2 cache is optional: main.py re-extracts on a miss. It must not
# re-extract into a read-only /kaggle/input, which would only fail AFTER the
# whole forward pass, so fall back to a writable directory instead. Since the
# milestone this notebook exists for is "the cache is already published",
# missing it here means the wrong dataset was attached, not a routine event --
# print loudly rather than silently eating a 90k-image forward pass.
vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
found = find_visual_cache(DATASET, SEED, vit_name, hint=FEATURE_DIR)
if found is not None:
    FEATURE_DIR = str(found)
    print("features cache:", FEATURE_DIR)
else:
    print(f"[features] WARNING: no cache found for {DATASET}/seed{SEED}/{vit_name}")
    print(f"  under {FEATURE_DIR!r} or the default Kaggle input roots.")
    print("  Falling back to extracting it in THIS session -- check that the")
    print("  extract_visual_features.ipynb output dataset is attached if that")
    print("  was not the intent.")
    FEATURE_DIR = "/kaggle/working/features"

if not str(FEATURE_DIR).startswith("/kaggle/input"):
    Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)
SAVE_DIR = Path(OUTPUT_DIR) / DATASET
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{SAMPLER}: {spec.passes} pass, prefix_exact={spec.prefix_exact} - {spec.why}")
print(f"GPUs visible: {visible_gpu_count()}")
print("config:", sampler_cfg)

In [ ]:
import time

BUDGETS = config["cumulative_budget"]
workers = visible_gpu_count() if PARALLEL else 1

# Budget sharding only helps a sampler whose budgets are independent runs.
# `spec.prefix_exact` is exactly that property, so it -- not a hand-kept list
# of names -- decides, and a new sampler cannot drift out of sync with it.
shard_budgets = SPLIT_BUDGETS and workers > 1 and not spec.prefix_exact and len(BUDGETS) > 1
if SPLIT_BUDGETS and spec.prefix_exact:
    print(f"[shard] {SAMPLER} is prefix-exact: one shared selection pass covers every")
    print("        budget, so its sweep stays on a single GPU (sharding would repeat")
    print("        that pass per shard). This is expected, not a misconfiguration.")

def budget_shards(budgets, n):
    """Deal budgets round-robin so each shard gets a mix of cheap and expensive.

    Cost grows with the budget, so a contiguous split would hand one worker
    every large budget and leave the other idle for most of the session.
    """
    groups = [budgets[i::n] for i in range(n)]
    return [g for g in groups if g]

# One configuration, so at most one run name -- split into budget shards only.
RUN = RUN_NAME or main._default_run_name(SAMPLER, sampler_cfg)
if SEED != config.get("random_seed", 42):
    RUN = f"{RUN}_s{SEED}"

base_kwargs = dict(
    data_path=str(data_path),
    sampler_name=SAMPLER,
    num_classes=dataset_info["num_classes"],
    data_descriptions=dataset_info.get("descriptions", {}),
    prompt_templates=config.get("prompt_templates", []),
    sampler_cfg=sampler_cfg,
    probe_epochs=training_cfg["probe_epochs"],
    probe_lr=training_cfg["probe_lr"],
    random_seed=SEED,
    save_dir=str(SAVE_DIR),
    verbose=True,
    model_cfg=config.get("models", {}),
    feature_cache_dir=FEATURE_DIR,
    mmap_cache_dir=MMAP_CACHE_DIR,
    run_name=RUN,
    device_string="cuda:0",
)

jobs = []
shard_tags = []
# Resume: a finished run wrote its merged results file. Re-running the notebook
# after a session timeout then costs nothing for what is already done.
if (SAVE_DIR / f"{RUN}_results.pt").is_file():
    print(f"already finished, nothing to do: {RUN}_results.pt")
elif shard_budgets:
    shards = budget_shards(BUDGETS, workers)
    shard_tags = [f"shard{i}" for i in range(len(shards))]
    for tag, budgets in zip(shard_tags, shards):
        jobs.append((f"{RUN}:{tag}", dict(
            base_kwargs, cumulative_budget=budgets, shard_tag=tag,
        )))
else:
    jobs.append((RUN, dict(base_kwargs, cumulative_budget=BUDGETS)))

print(f"run: {RUN}")
for label, kwargs in jobs:
    print(f"   {label:40} budgets={kwargs['cumulative_budget']}")

# Export the .npz to memory-mappable .npy files ONCE, here in the parent.
#
# Must be the parent: two GPU workers exporting the same ~15 GiB concurrently
# would race on the same filenames. Once exported, each worker maps the same
# pages and the OS page cache serves them all from one copy, so a second worker
# costs no additional pixel memory.
#
# ImageFolder datasets skip this: they hold a path list and read per file, so
# they were never the problem.
if str(data_path).endswith(".npz"):
    from data.npz_mmap import export_npz_to_npy

    export_npz_to_npy(str(data_path), MMAP_CACHE_DIR)
    print(f"mmap export ready: {MMAP_CACHE_DIR}")
else:
    print("ImageFolder dataset: reads per file, no mmap export needed")

started = time.time()
results = run_variants_parallel(jobs, main.run_on_worker, num_workers=workers)

print("=" * 70)
for result in results:
    status = "ok" if result["ok"] else "FAILED"
    print(f"{result['label']:40} {status:8} {format_duration(result['seconds'])}")
failed = [r["label"] for r in results if not r["ok"]]
if results:
    print(f"total {format_duration(time.time() - started)} | "
          f"{len(results) - len(failed)}/{len(results)} succeeded")
assert not failed, f"jobs failed: {failed}"

# Fold the shards back into the one `<run>_results.pt` an unsharded run would
# have written, so nothing downstream needs to know this run was split.
if shard_tags:
    linear = main.merge_budget_shards(str(SAVE_DIR), RUN, shard_tags)
    print(f"[merge] {RUN}: {len(linear)} budgets -> {RUN}_results.pt")

In [ ]:
# Results table, in budget order.
#
# The dispatch cell above runs two GPUs in one output stream, so their progress
# lines interleave and a budget's numbers can land anywhere. This cell ignores
# all of that and reads the saved `<run>_results.pt`, which is the authoritative
# record -- so the table below is correctly ordered no matter how the logs came
# out, and re-running this cell alone reprints it without recomputing anything.
#
# Metrics only. Per-step acquisition scores, sigma and the sanity report live in
# the saved files for later analysis; printing them here would recreate exactly
# the wall of text this cell exists to avoid.
import torch

results_path = SAVE_DIR / f"{RUN}_results.pt"
assert results_path.is_file(), (
    f"no results at {results_path} -- the run above did not finish"
)
payload = torch.load(results_path, weights_only=False)
linear = payload["linear"]

print(f"{payload['sampler']}  |  {payload['dataset']}  |  seed {payload['seed']}")
print(f"run_name: {payload['run_name']}")
if payload.get("sharded_over"):
    print(f"budget shards: {', '.join(payload['sharded_over'])}")
print()

header = f"{'budget':>8}  {'accuracy':>9}  {'precision':>9}  {'recall':>9}  {'macro F1':>9}  {'select s':>9}"
print(header)
print("-" * len(header))
for budget in sorted(linear):
    row = linear[budget]
    print(f"{budget:>8}  {row['acc']:>9.4f}  {row['precision']:>9.4f}  "
          f"{row['recall']:>9.4f}  {row['f1']:>9.4f}  {row['selection_seconds']:>9.1f}")
print("-" * len(header))

best = max(linear, key=lambda b: linear[b]["acc"])
print(f"best accuracy {linear[best]['acc']:.4f} at budget {best}")

# A sanity severity worse than "ok" means the selection itself looked
# degenerate at some budget -- worth seeing next to the numbers rather than
# buried in the log above.
worst = {b: linear[b].get("sanity_severity", "ok") for b in sorted(linear)}
flagged = {b: s for b, s in worst.items() if s != "ok"}
if flagged:
    print(f"sanity: {flagged}  <- check the run log for details")
else:
    print("sanity: ok at every budget")

In [ ]:
# Package the results as ONE zip at the top of /kaggle/working, then delete the
# loose files -- the same shape the two extraction notebooks use, and for the
# same reason.
#
# Kaggle's Output tab lists what is left in /kaggle/working when the session
# ends, and in a "Save & Run All" session that is the ONLY way to get a file
# out: there is no terminal and no kaggle CLI, so printing `kaggle datasets
# create` commands is advice nobody in that session can follow. The zip goes to
# the top level where it is easy to find, and the loose checkpoints are removed
# once it exists: keeping both means downloading everything twice, and a
# session over the ~20 GB Output quota shows NOTHING at all, including the
# files that were fine.
import shutil

from utils import results_archive_stem

# Delete the .npy mmap export BEFORE archiving.
#
# It is scratch for the pixel-reading pass only -- nothing below it, and
# nothing downstream, ever reads it again. But it is the single largest thing
# in /kaggle/working: PathMNIST-224 exports ~15 GiB, and a real run of this
# notebook finished with 16.1 GB sitting in Output, ~15.7 GB of it this
# directory. That is 80% of the ~20 GB Output quota spent on a temporary
# file, and a session that goes over the quota shows NOTHING in the Output
# tab -- including the zip that was fine. Same failure the nucleus run hit
# (CLAUDE.md: "The nucleus run dies of DISK, not memory").
if MMAP_CACHE_DIR and Path(MMAP_CACHE_DIR).is_dir():
    freed = sum(f.stat().st_size for f in Path(MMAP_CACHE_DIR).rglob("*") if f.is_file())
    shutil.rmtree(MMAP_CACHE_DIR, ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")
SOURCE = Path(OUTPUT_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
# make_archive must not write inside the directory being archived, or it packs
# a partial copy of itself. OUTPUT_DIR is a subdirectory of WORKING, so writing
# to WORKING is safe -- assert it rather than assume it.
assert SOURCE.resolve() != WORKING.resolve(), (
    "OUTPUT_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

# dataset + sampler + seed are exactly the axes that make two result sets
# non-interchangeable, so they are the name.
STEM = results_archive_stem(DATASET, SAMPLER, SEED)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

# The zip root holds `<dataset>/`, which is the layout
# evaluate_al_sampler.ipynb expects after Kaggle extracts an uploaded dataset.
print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file():
        print(f"    {path.relative_to(SOURCE)}  ({path.stat().st_size / 1e6:.2f} MB)")

# The zip is written and its size is known, so the originals are redundant.
shutil.rmtree(SOURCE, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
     Kaggle extracts it into a directory named after the zip, so the
     checkpoints end up one level down. That is expected.
  3. In evaluate_al_sampler.ipynb: Add Data -> your new dataset, then point
     CHECKPOINT_ROOT at it to rebuild the table and fit PALM.""")